# IEEE-CIS fraud data: exploratory analysis

Class imbalance, and how many transactions share each card, address, email domain or device.
These numbers justify the graph-building limits in `src/data/build_graph.py`.

In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
data_dir = '../data/raw'
transaction_file = os.path.join(data_dir, 'train_transaction.csv')
if not os.path.exists(transaction_file):
    raise FileNotFoundError(f"{transaction_file} not found. Download the IEEE-CIS data from Kaggle first.")

df = pd.read_csv(transaction_file, usecols=['TransactionID', 'TransactionDT', 'TransactionAmt', 'isFraud',
                                             'card1', 'card2', 'addr1', 'addr2', 'P_emaildomain'])
print(f"Loaded {len(df)} transactions.")

## 1. Class imbalance

In [ ]:
counts = df['isFraud'].value_counts().sort_index()
pct = df['isFraud'].value_counts(normalize=True).sort_index() * 100
print("Counts:\n", counts)
print("\nPercentages:\n", pct.round(3))
print(f"\nFraud rate: {df['isFraud'].mean():.4f}")

plt.bar(counts.index.astype(str), counts.values)
plt.title('Class imbalance: isFraud')
plt.xlabel('isFraud (0 = legitimate, 1 = fraud)')
plt.ylabel('Transactions')
plt.show()

## 2. How many transactions share each entity

Graph edges link transactions that share a value. A value shared by very many transactions
would create very many edges, so the graph builder caps neighbours per node.

In [ ]:
relation_cols = ['card1', 'card2', 'addr1', 'addr2', 'P_emaildomain']
rows = []
for col in relation_cols:
    counts = df[col].value_counts()
    rows.append({
        'column': col,
        'distinct_values': len(counts),
        'most_common_count': int(counts.max()),
        'values_seen_over_100_times': int((counts > 100).sum()),
        'rows_missing': int(df[col].isna().sum()),
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

**Reading the table:** `addr2` has very few distinct values, so almost every transaction shares it.
That is why `build_graph.py` excludes it. Columns with a large most-common count are the ones
where neighbour capping matters.